In [1]:
!pip install reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 17.8 MB/s  0:00:00


In [2]:
import json
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import inch
from reportlab.lib import colors
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image, PageBreak
)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER, TA_LEFT
import os

with open("../outputs/final_career_report.json", "r", encoding="utf-8") as f:
    report = json.load(f)

print("Final report data loaded")
print(f"Report generated on: {report['generated_on']}")

Final report data loaded
Report generated on: 2026-07-23 06:02:53


In [10]:
styles = getSampleStyleSheet()

title_style = ParagraphStyle('CustomTitle', parent=styles['Heading1'], fontSize=24, textColor=colors.HexColor('#1a1a2e'), spaceAfter=6, alignment=TA_CENTER)
subtitle_style = ParagraphStyle('CustomSubtitle', parent=styles['Normal'], fontSize=11, textColor=colors.HexColor('#666666'), alignment=TA_CENTER, spaceAfter=20)
section_style = ParagraphStyle('SectionHeader', parent=styles['Heading2'], fontSize=15, textColor=colors.HexColor('#0f3460'), spaceBefore=16, spaceAfter=8)
body_style = ParagraphStyle('Body', parent=styles['Normal'], fontSize=10, leading=15, spaceAfter=6)
score_style = ParagraphStyle('ScoreStyle', parent=styles['Normal'], fontSize=32, textColor=colors.HexColor('#16a34a'), alignment=TA_CENTER, spaceBefore=10, spaceAfter=10, leading=40)
print("PDF styles defined")

PDF styles defined


In [13]:
doc = SimpleDocTemplate("../outputs/Cognitive Nexus.pdf", pagesize=A4,
                         topMargin=0.6*inch, bottomMargin=0.6*inch,
                         leftMargin=0.7*inch, rightMargin=0.7*inch)

elements = []

# Title
elements.append(Paragraph("🚀 Cognitive Nexus", title_style))
elements.append(Paragraph(f"Career Readiness Report — Generated {report['generated_on']}", subtitle_style))

elements.append(Paragraph("Career Readiness Score", section_style))
elements.append(Spacer(1, 0.3*inch))
score_text = f"{report['career_readiness']['career_readiness_score']} / 100"
elements.append(Paragraph(score_text, score_style))
elements.append(Spacer(1, 0.25*inch))

# Breakdown table
breakdown_data = [["Category", "Score"]]
for k, v in report['career_readiness']['breakdown'].items():
    breakdown_data.append([k.title(), str(v)])

breakdown_table = Table(breakdown_data, colWidths=[3*inch, 2*inch])
breakdown_table.setStyle(TableStyle([
    ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#0f3460')),
    ('TEXTCOLOR', (0,0), (-1,0), colors.white),
    ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
    ('ALIGN', (0,0), (-1,-1), 'CENTER'),
    ('GRID', (0,0), (-1,-1), 0.5, colors.grey),
    ('ROWBACKGROUNDS', (0,1), (-1,-1), [colors.white, colors.HexColor('#f0f0f0')]),
    ('FONTSIZE', (0,0), (-1,-1), 10),
    ('TOPPADDING', (0,0), (-1,-1), 6),
    ('BOTTOMPADDING', (0,0), (-1,-1), 6),
]))
elements.append(breakdown_table)
elements.append(Spacer(1, 0.2*inch))

# Add readiness chart if it exists
chart_path = "../outputs/chart_readiness_breakdown.png"
if os.path.exists(chart_path):
    elements.append(Image(chart_path, width=5.5*inch, height=3*inch))
    elements.append(Spacer(1, 0.1*inch))

# ATS Score
elements.append(Paragraph("ATS Analysis", section_style))
elements.append(Paragraph(f"<b>ATS Score:</b> {report['ats_analysis']['ats_score']} / 100", body_style))
for fb in report['ats_analysis']['feedback']:
    elements.append(Paragraph(f"• {fb}", body_style))

elements.append(PageBreak())

# Skill Gap Analysis
elements.append(Paragraph("Skill Gap Analysis", section_style))
elements.append(Paragraph(f"<b>Target Role:</b> {report['skill_gap_analysis']['target_role']}", body_style))
elements.append(Paragraph(f"<b>Skill Match:</b> {report['skill_gap_analysis']['match_percentage']}%", body_style))
elements.append(Paragraph(f"<b>Missing Skills:</b> {', '.join(report['skill_gap_analysis']['missing_skills'])}", body_style))

chart_path2 = "../outputs/chart_skill_match.png"
if os.path.exists(chart_path2):
    elements.append(Spacer(1, 0.1*inch))
    elements.append(Image(chart_path2, width=3.5*inch, height=3.5*inch))

elements.append(Spacer(1, 0.2*inch))

# Learning Roadmap
elements.append(Paragraph(f"Learning Roadmap ({report['learning_roadmap']['total_weeks']} weeks)", section_style))
roadmap_data = [["Duration", "Skill", "Resource"]]
for item in report['learning_roadmap']['plan']:
    roadmap_data.append([item['duration'], item['skill'].title(), item['resource']])

roadmap_table = Table(roadmap_data, colWidths=[1.2*inch, 1.5*inch, 3.3*inch])
roadmap_table.setStyle(TableStyle([
    ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#0f3460')),
    ('TEXTCOLOR', (0,0), (-1,0), colors.white),
    ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
    ('GRID', (0,0), (-1,-1), 0.5, colors.grey),
    ('ROWBACKGROUNDS', (0,1), (-1,-1), [colors.white, colors.HexColor('#f0f0f0')]),
    ('FONTSIZE', (0,0), (-1,-1), 8),
    ('VALIGN', (0,0), (-1,-1), 'TOP'),
    ('TOPPADDING', (0,0), (-1,-1), 5),
    ('BOTTOMPADDING', (0,0), (-1,-1), 5),
]))
elements.append(roadmap_table)

elements.append(PageBreak())

# Resume Improvement Suggestions (LLM section)
elements.append(Paragraph("AI-Generated Resume Improvement Analysis", section_style))
llm_feedback = report['resume_improvement_suggestions'].get('llm_generated_feedback', 'N/A')
for line in llm_feedback.split('\n'):
    if line.strip():
        elements.append(Paragraph(line.strip(), body_style))

elements.append(Spacer(1, 0.2*inch))

# Interview Prep (LLM section)
elements.append(Paragraph("AI-Personalized Interview Questions", section_style))
interview_prep = report['interview_preparation']
llm_questions = interview_prep.get('llm_personalized_questions', 'N/A')
for line in llm_questions.split('\n'):
    if line.strip():
        elements.append(Paragraph(line.strip(), body_style))

doc.build(elements)
print("PDF report generated: ../outputs/Cognitive Nexus_Report.pdf")

PDF report generated: ../outputs/CareerForge_AI_Report.pdf


In [14]:
pdf_path = "../outputs/Cognitive Nexus_Report.pdf"
if os.path.exists(pdf_path):
    size_kb = os.path.getsize(pdf_path) / 1024
    print(f"PDF successfully created: {pdf_path}")
    print(f"File size: {size_kb:.1f} KB")
    print("\nNotebook 13 (PDF Report Generator) — COMPLETE")
else:
    print("PDF generation failed")

PDF successfully created: ../outputs/Cognitive Nexus_Report.pdf
File size: 99.5 KB

Notebook 13 (PDF Report Generator) — COMPLETE
